In [10]:
import numpy as np
from sklearn.model_selection import train_test_split
import datetime
from keras.datasets import fashion_mnist
import wandb

In [11]:
%load_ext autoreload
%autoreload 2
from Model import NeuralNetwork

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
def normalize(x):
    return x.reshape(len(x), -1).astype('float64') / (np.max(x) - np.min(x))

In [13]:
def load_and_prepare_data(dataset="fashion_mnist"):
    # Load the Fashion MNIST dataset
    (x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
    
    # Using train_test_split to separate validation data (10% of training data)
    x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.1, random_state=69)
    
    # Normalize image pixel values
    x_train = normalize(x_train)
    x_val   = normalize(x_val)
    x_test  = normalize(x_test)
    
    # Determine the number of classes from the unique labels
    classes = np.unique(y_train)
    num_classes = len(classes)
    
    # One-hot encode labels based on the discovered number of classes
    y_train = np.eye(num_classes)[y_train]
    y_val   = np.eye(num_classes)[y_val]
    y_test  = np.eye(num_classes)[y_test]
    
    return x_train, y_train, x_val, y_val, x_test, y_test

In [14]:
def train_and_evaluate(config=None):
    with wandb.init(config=config):
        cfg = wandb.config
        # Create a dynamic name based on hyperparameters
        sweep_name = f"hl_{cfg.num_layers}_hs_{cfg.hidden_size}_bs_{cfg.batch_size}_ac_{cfg.activation}_opt_{cfg.optimizer}_lr_{cfg.learning_rate}"
    
        # Assign the dynamically generated name
        wandb.run.name = sweep_name
        wandb.run.save()
        # Load and prepare the dataset
        x_train, y_train, x_val, y_val, x_test, y_test = load_and_prepare_data()
        
        # Model with configuration parameters
        model = NeuralNetwork(
            input_size = x_train.shape[1],
            num_classes = y_train.shape[1],
            num_hidden = cfg.num_layers,
            hidden_units = cfg.hidden_size,
            init_method = cfg.weight_init,
            activation = cfg.activation,
            loss_fn = cfg.loss,
            epochs = cfg.epochs,
            batch_size = cfg.batch_size,
            optimizer = cfg.optimizer,
            lr = cfg.learning_rate,
            weight_decay = cfg.weight_decay,
            momentum = cfg.momentum if hasattr(cfg, 'momentum') else 0.9,
            beta = cfg.beta if hasattr(cfg, 'beta') else 0.9,
            beta1 = cfg.beta1 if hasattr(cfg, 'beta1') else 0.9,
            beta2 = cfg.beta2 if hasattr(cfg, 'beta2') else 0.999,
            epsilon = cfg.epsilon if hasattr(cfg, 'epsilon') else 1e-6,
            iswandb = True
        )
        
        # Train the model using the training and validation data
        model.fit(x_train, y_train, x_val, y_val)
        
        # Evaluate on validation set
        val_preds = model.predict(x_val.T)
        val_loss  = model.compute_loss(val_preds, y_val)
        val_acc   = model.accuracy(val_preds, y_val)
        
        # Evaluate on test set
        test_preds = model.predict(x_test.T)
        test_loss  = model.compute_loss(test_preds, y_test)
        test_acc   = model.accuracy(test_preds, y_test)
        
        # Log evaluation metrics to wandb with a timestamp
        wandb.log({
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "test_loss": test_loss,
            "test_accuracy": test_acc,
            "created": datetime.datetime.now().isoformat()
        })

In [15]:
sweep_config = {
    'method': 'bayes',
    'name': 'Bayesian_sweep_cross_entropy',
    'metric': {'name': 'validation_accuracy', 'goal': 'maximize'},
    'parameters': {
        'epochs': {'values': [5, 10]},
        'num_layers': {'values': [3, 4, 5]},
        'hidden_size': {'values': [32, 64, 128]},
        'weight_decay': {'values': [0, 0.0005, 0.5]},
        'learning_rate': {'values': [0.001, 0.0001]},
        'optimizer': {'values': ['sgd', 'momentum', 'nag', 'rmsprop', 'adam', 'nadam']},
        'batch_size': {'values': [16, 32, 64]},
        'weight_init': {'values': ['Random', 'Xavier']},
        'activation': {'values': ['Sigmoid', 'Tanh', 'ReLU']},
        'loss': {'values': ['cross_entropy']}
    }
}

In [16]:
def run_experiment():
    sweep_id = wandb.sweep(sweep_config, project="fashion-mnist-classification")
    wandb.agent(sweep_id, function=train_and_evaluate, count=10)
    wandb.finish()

In [ ]:
if __name__ == "__main__":
    run_experiment()

Create sweep with ID: 5f8m2t1u
Sweep URL: https://wandb.ai/mrsagarbiswas-iit-madras/fashion-mnist-classification/sweeps/5f8m2t1u


wandb: Agent Starting Run: oceux43d with config:
wandb: 	activation: Tanh
wandb: 	batch_size: 32
wandb: 	epochs: 10
wandb: 	hidden_size: 128
wandb: 	learning_rate: 0.0001
wandb: 	loss: cross_entropy
wandb: 	num_layers: 4
wandb: 	optimizer: adam
wandb: 	weight_decay: 0
wandb: 	weight_init: Xavier
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


Epoch 1: train_loss = 0.44, valid_loss = 0.45, train_accuracy = 0.84, val_accuracy = 0.84
Epoch 2: train_loss = 0.39, valid_loss = 0.41, train_accuracy = 0.86, val_accuracy = 0.86


wandb: WARNING Fatal error while uploading data. Some run data will not be synced, but it will still be written to disk. Use `wandb sync` at the end of the run to try uploading.


Epoch 3: train_loss = 0.36, valid_loss = 0.39, train_accuracy = 0.87, val_accuracy = 0.86
Epoch 4: train_loss = 0.34, valid_loss = 0.38, train_accuracy = 0.88, val_accuracy = 0.87
Epoch 5: train_loss = 0.33, valid_loss = 0.37, train_accuracy = 0.88, val_accuracy = 0.87
Epoch 6: train_loss = 0.32, valid_loss = 0.36, train_accuracy = 0.89, val_accuracy = 0.88
Epoch 7: train_loss = 0.31, valid_loss = 0.35, train_accuracy = 0.89, val_accuracy = 0.88
Epoch 8: train_loss = 0.30, valid_loss = 0.35, train_accuracy = 0.89, val_accuracy = 0.88
Epoch 9: train_loss = 0.29, valid_loss = 0.34, train_accuracy = 0.90, val_accuracy = 0.88
Epoch 10: train_loss = 0.28, valid_loss = 0.34, train_accuracy = 0.90, val_accuracy = 0.88
